# 06 — Leave-One-Turbine-Out Evaluation (Phase 2)

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent) if Path.cwd().name=='notebooks' else str(Path.cwd()))
from src.utils.data_loader import DatasetSource
from src.ml.loto import run_loto
import pandas as pd

src = DatasetSource()
ei = src.read_csv('Wind Farm A/event_info.csv', sep=';')
FEATURES = ['sensor_0_avg','wind_speed_3_avg','sensor_5_avg','power_30_avg','sensor_11_avg','sensor_13_avg']

res = run_loto(src, 'A', FEATURES, ei, models=['zscore','iforest'], if_contaminations=[0.05,0.10,0.15], z_threshold=2.0)
res.to_csv('reports/loto_results.csv', index=False)
print(res.groupby(['model','contamination'])[['precision','recall','f1','false_alarms_per_day','event_detection_rate']].mean().to_string())


In [ ]:
# Per-turbine summary
pivot = res.pivot_table(index='test_asset', columns='model', values='f1', aggfunc='mean')
print(pivot.to_string())


In [ ]:
# Visualize F1 by turbine and model
import matplotlib.pyplot as plt
fig, ax = plt.subplots(figsize=(8,4))
for model, g in res.groupby('model'):
    ax.plot(g['test_asset'].astype(str), g['f1'], marker='o', label=model)
ax.set_title('LOTO F1 by test turbine')
ax.set_xlabel('test asset'); ax.set_ylabel('F1'); ax.legend()
plt.tight_layout(); plt.show()
